# Notebook G — QMKL avec sous-ensembles de features
## Couvrir un espace de grande dimension avec des kernels à Q=4 qubits

**Question centrale** : Un circuit quantique à Q=4 qubits ne peut encoder que 4 features.
Comment exploiter un dataset à d=20 ou d=30 features ?

**Réponse** : chaque kernel K_m voit un **sous-ensemble différent** de 4 features.
L'ensemble combiné couvre tout l'espace — et QMKL apprend lesquels comptent.

**Plan :**
1. Le problème des 4 qubits — formule mathématique et espace implicite
2. Stratégies de sélection des sous-ensembles + dataset Breast Cancer
3. Calcul analytique des kernels sur chaque sous-ensemble
4. Centered Alignment — quelles features comptent vraiment ?
5. Comparaison AUC : baseline (4 fixes) vs subsets différents
6. Importance implicite des features : ce que QMKL révèle


## Section 1 — Le problème des 4 qubits

### Contrainte matérielle

Un circuit quantique à $Q$ qubits encode exactement $Q$ features réelles.
Pour $Q=4$, la feature map quantique est :

$$\phi_Q : x \in \mathbb{R}^Q \longrightarrow |\psi(x)\rangle \in \mathbb{C}^{2^Q}$$

Le kernel quantique induit est :
$$K(x, x') = |\langle \psi(x) | \psi(x') \rangle|^2 \in [0, 1]$$

Avec $Q=4$ : espace de Hilbert de dimension $2^4 = 16$.

**Problème** : les datasets réels ont $d \gg Q$ features (German Credit : $d=20$,
Breast Cancer : $d=30$, Bank Marketing : $d=16$).
Dans l'implémentation actuelle, on prend toujours les 4 premières composantes PCA.
**Ce choix est arbitraire et ignore la majorité des informations.**

---

### Solution : feature-subset kernels

Pour chaque kernel $m$, on définit un sous-ensemble $S_m \subset \{1,\ldots,d\}$ avec $|S_m| = Q$ :

$$\boxed{K_m(x, x') = K_{\text{quantum}}\!\left(x[S_m],\, x'[S_m]\right)}$$

QMKL combine ensuite :

$$K_{\text{mkl}}(x, x') = \sum_{m=1}^{M} w_m \cdot K_m(x, x')
= \sum_{m=1}^{M} w_m \cdot K_{\text{quantum}}(x[S_m], x'[S_m])$$

---

### L'espace implicite de grande dimension

Chaque $K_m$ admet une feature map $\phi_m : \mathbb{R}^Q \to \mathbb{R}^{2^Q}$.
Le kernel combiné correspond à la **somme directe** des espaces :

$$\Phi_{\text{mkl}}(x) = \bigoplus_{m=1}^{M} \sqrt{w_m}\,\phi_m(x[S_m])
\in \mathbb{R}^{M \times 2^Q}$$

| Paramètres | Dimension de l'espace implicite |
|---|---|
| $M=7$ subsets, $Q=4$ | $7 \times 16 = 112$ |
| $M=22$ subsets, $Q=4$ | $22 \times 16 = 352$ |
| Toutes combos $\binom{30}{4}$ subsets | $27\,405 \times 16 \approx 438\,000$ |

**Propriété clé** : somme pondérée (poids $w_m \geq 0$) de kernels SDP = SDP. ✓
Donc $K_{\text{mkl}}$ est toujours un kernel valide.


## Section 2 — Dataset et stratégies de sous-ensembles

On utilise **Breast Cancer Wisconsin** (sklearn) : $N=569$, $d=30$ features, classification binaire.
C'est le dataset idéal pour démontrer l'idée : 30 features >> 4 qubits.

### Trois stratégies de subsets

| Stratégie | M | Description |
|---|---|---|
| **Non-overlapping** | 7 | Groupes disjoints : {0-3}, {4-7}, ..., {24-27} — couvre 28/30 features |
| **Random overlapping** | 15 | Tirages aléatoires — capture les interactions cross-groupes |
| **Baseline actuelle** | 1 | Toujours features {0,1,2,3} — comme l'implémentation actuelle |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

np.random.seed(42)
plt.rcParams.update({'font.family': 'sans-serif', 'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

# ── Chargement et préprocessing ────────────────────────────────────────────
data = load_breast_cancer()
X_raw, y = data.data, data.target      # y : 0=malignant, 1=benign
feature_names = list(data.feature_names)
N, d = X_raw.shape
print(f"Dataset Breast Cancer : N={N}, d={d} features")
print(f"Classes : {dict(zip(['malignant','benign'], np.bincount(y)))}")

# Standardiser puis scaler vers [0, 2pi] (encodage quantique)
X_std = StandardScaler().fit_transform(X_raw)
X = MinMaxScaler(feature_range=(0, 2*np.pi)).fit_transform(X_std)

# ── Génération des sous-ensembles ──────────────────────────────────────────
# 1) Non-overlapping : groupes de 4 features consécutives (features 0 à 27)
non_overlap = [list(range(4*i, 4*i+4)) for i in range(7)]

# 2) Random overlapping : 15 subsets aléatoires distincts
rng = np.random.RandomState(42)
already = set(tuple(s) for s in non_overlap)
random_subsets = []
while len(random_subsets) < 15:
    s = tuple(sorted(rng.choice(d, size=4, replace=False)))
    if s not in already:
        already.add(s)
        random_subsets.append(list(s))

all_subsets = non_overlap + random_subsets    # 22 subsets au total
print(f"\nNon-overlapping : {len(non_overlap)} subsets (features 0-27)")
print(f"Random overlapping : {len(random_subsets)} subsets")
print(f"Total : {len(all_subsets)} subsets")

# ── Visualisation de la couverture ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
matrix = np.zeros((len(all_subsets), d))
for i, s in enumerate(all_subsets):
    matrix[i, s] = 1.0

im = ax.imshow(matrix, cmap='Blues', aspect='auto', vmin=0, vmax=1.3)
ax.axhline(6.5, color='#e74c3c', lw=2.5, linestyle='--', alpha=0.8)
ax.axvline(27.5, color='orange', lw=1.5, linestyle=':', alpha=0.8, label='Feature 28-29 (exclues des non-overlap)')
ax.set_xlabel('Feature index (0-29)', fontsize=11)
ax.set_ylabel('Sous-ensemble $S_m$', fontsize=11)
ax.set_yticks(range(len(all_subsets)))
ax.set_yticklabels([f'S{i+1}' for i in range(len(all_subsets))], fontsize=8)
ax.set_title('Couverture des features par sous-ensemble\n(bleu = feature incluse dans ce subset)', fontweight='bold')
ax.text(-2.5, 3, 'Non-\noverlap', fontsize=9, color='#2980b9', ha='center', va='center')
ax.text(-2.5, 14, 'Random\noverlap', fontsize=9, color='#e74c3c', ha='center', va='center')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

# Afficher les 3 premiers non-overlapping et 3 random
for i, s in enumerate(all_subsets[:3]):
    print(f"S{i+1} (non-overlap): features {s} → {[feature_names[k][:18] for k in s]}")
print("...")
for i, s in enumerate(all_subsets[7:10], start=8):
    print(f"S{i} (random):       features {s}")


## Section 3 — Kernels quantiques analytiques sur chaque sous-ensemble

On implémente les formules analytiques de $K_Z$ et $K_{ZZ}$ pour $Q=4$ features,
**sans Qiskit** — juste numpy.

$$K_Z(x, x') = \prod_{k=1}^{Q} \cos^2\!\left(\alpha(x_k - x'_k)\right)$$

$$K_{ZZ}(x, x') = K_Z(x, x') \times \prod_{k < l} \cos^2\!\left(\alpha(x_k x_l - x'_k x'_l)\right)$$

**Paramètres** : 2 familles (Z, ZZ) × 2 valeurs de $\alpha$ (0.5, 2.0)
= **4 kernels par subset** $\times$ 22 subsets = **88 kernels au total**.

Chaque kernel $K_m$ est calculé sur les $N \times N$ paires du dataset,
en utilisant uniquement les 4 features de $S_m$.


In [ ]:
# ── Formules analytiques des kernels quantiques ───────────────────────────

def K_Z(X1, X2, alpha=1.0):
    '''K_Z(x,x') = prod_k cos^2(alpha*(x_k - x'_k))
    X1: (n1, Q), X2: (n2, Q) — Q features seulement.
    Retourne (n1, n2).'''
    n1, Q = X1.shape
    n2 = X2.shape[0]
    K = np.ones((n1, n2))
    for k in range(Q):
        diff = X1[:, k:k+1] - X2[:, k].reshape(1, -1)
        K *= np.cos(alpha * diff)**2
    return K

def K_ZZ(X1, X2, alpha=1.0):
    '''K_ZZ = K_Z * prod_{k<l} cos^2(alpha*(x_k*x_l - x'_k*x'_l))
    Ajoute les termes d interaction pairwise au-dessus de K_Z.'''
    K = K_Z(X1, X2, alpha)
    Q = X1.shape[1]
    for k in range(Q):
        for l in range(k+1, Q):
            interact1 = (X1[:, k] * X1[:, l]).reshape(-1, 1)
            interact2 = (X2[:, k] * X2[:, l]).reshape(1, -1)
            K *= np.cos(alpha * (interact1 - interact2))**2
    return K

def normalize_kernel(K):
    '''Normalisation: K[i,j] /= sqrt(K[i,i] * K[j,j]) pour K dans [0,1].'''
    diag = np.sqrt(np.diag(K))
    return K / (np.outer(diag, diag) + 1e-12)

# ── Calcul de tous les kernels ─────────────────────────────────────────────
FAMILIES = {
    'Z_a05':  (K_Z,  0.5),
    'Z_a20':  (K_Z,  2.0),
    'ZZ_a05': (K_ZZ, 0.5),
    'ZZ_a20': (K_ZZ, 2.0),
}

kernels = {}         # nom → matrice (N, N)
kernel_meta = {}     # nom → (subset_idx, famille, alpha)

print("Calcul des kernels...", end=' ')
for s_idx, subset in enumerate(all_subsets):
    X_sub = X[:, subset]
    for fname, (fn, alpha) in FAMILIES.items():
        name = f"S{s_idx+1}_{fname}"
        K = fn(X_sub, X_sub, alpha=alpha)
        kernels[name] = normalize_kernel(K)
        kernel_meta[name] = (s_idx, fname, alpha)
print(f"done. {len(kernels)} kernels calculés.")

# ── Visualiser 4 heatmaps : même features S1, 4 familles ──────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
n_show = 50    # afficher 50x50 pour lisibilité

for ax, (fname, (fn, alpha)) in zip(axes, FAMILIES.items()):
    name = f"S1_{fname}"
    Kv = kernels[name][:n_show, :n_show]
    im = ax.imshow(Kv, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'{fname}\nfeatures {all_subsets[0]}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Sample')
    ax.set_ylabel('Sample')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Heatmaps $K(x,x')$ — Sous-ensemble S1 = {features 0,1,2,3}, 4 configurations",
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Comparaison: même subset S1 vs subset S4 (features 12-15) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, s_idx, title in zip(axes, [0, 3],
                             ['S1: features {0,1,2,3}\n(baseline actuelle)',
                              'S4: features {12,13,14,15}\n(autre groupe)']):
    name = f"S{s_idx+1}_ZZ_a20"
    K_show = kernels[name][:n_show, :n_show]
    im = ax.imshow(K_show, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'$K_{{ZZ}}$ (α=2.0) — {title}', fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Même kernel ZZ, features différentes → structures de similarité différentes', fontweight='bold')
plt.tight_layout()
plt.show()
print("Les deux heatmaps ont des structures distinctes : les sous-ensembles voient des aspects différents des données.")


## Section 4 — Centered Alignment : quelles features comptent ?

On utilise le **Centered Kernel-Target Alignment** pour quantifier à quel point
chaque kernel est aligné avec la structure de classe.

### Rappel théorique

L'alignment centré entre kernel $K_m$ et cible $\mathbf{y}\mathbf{y}^\top$ :

$$\hat{A}(K_m,\, \mathbf{y}\mathbf{y}^\top)
= \frac{\langle \tilde{K}_m,\, \tilde{Y} \rangle_F}{\|\tilde{K}_m\|_F\, \|\tilde{Y}\|_F}$$

où $\tilde{K} = HKH$ est le kernel centré ($H = I - \frac{1}{n}\mathbf{1}\mathbf{1}^\top$).

### Poids optimaux

Les poids $w^* = S^{-1}q$ (voir Notebook C), avec :
- $S_{mn} = \hat{A}(K_m, K_n)$ — alignment mutuel (redondance)
- $q_m = \hat{A}(K_m, \mathbf{y}\mathbf{y}^\top)$ — alignment avec la cible

Un **poids élevé** = le kernel est bien aligné avec les classes ET diversifié
par rapport aux autres. QMKL effectue implicitement une **sélection de features**.


In [ ]:
# ── Centered Alignment ────────────────────────────────────────────────────

n = N
H = np.eye(n) - np.ones((n, n)) / n    # matrice de centrage (N, N)

y_pm = 2*y - 1          # {0,1} → {-1,+1}
Y = np.outer(y_pm, y_pm)
Yc = H @ Y @ H          # cible centrée (calculée une seule fois)

def centered_alignment(K, Yc=Yc, H=H):
    '''Alignment centré entre kernel K et cible Yc (déjà centrée).'''
    Kc = H @ K @ H
    num = np.sum(Kc * Yc)
    denom = np.sqrt(np.sum(Kc**2) * np.sum(Yc**2))
    return float(num / (denom + 1e-12))

# Alignment de chaque kernel avec la cible
kernel_names = list(kernels.keys())
M_total = len(kernel_names)
print(f"Calcul de l'alignment pour {M_total} kernels...", end=' ')
q = np.array([centered_alignment(kernels[name]) for name in kernel_names])
print("done.")
print(f"Alignment : min={q.min():.4f}, max={q.max():.4f}, mean={q.mean():.4f}")

# ── Organiser par subset et par famille ────────────────────────────────────
n_sub = len(all_subsets)
n_fam = len(FAMILIES)
fam_names = list(FAMILIES.keys())

# align_matrix[s_idx, f_idx] = alignment du kernel (subset s, famille f)
align_matrix = np.zeros((n_sub, n_fam))
for i, name in enumerate(kernel_names):
    s_idx, fname, alpha = kernel_meta[name]
    f_idx = fam_names.index(fname)
    align_matrix[s_idx, f_idx] = q[i]

# Alignment moyen par subset
align_by_subset = align_matrix.mean(axis=1)
best_subset = np.argmax(align_by_subset)

# ── Figure 1 : alignment par subset (barres) ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax = axes[0]
colors = ['#3498db']*7 + ['#e74c3c']*15
ax.bar(range(n_sub), align_by_subset, color=colors, edgecolor='white', linewidth=0.5)
ax.axvline(6.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Sous-ensemble $S_m$', fontsize=11)
ax.set_ylabel('Alignment moyen avec la cible', fontsize=11)
ax.set_title('Alignment par sous-ensemble\n(bleu=non-overlap, rouge=random)', fontweight='bold')
ax.set_xticks(range(n_sub))
ax.set_xticklabels([f'S{i+1}' for i in range(n_sub)], rotation=45, fontsize=8)
ax.annotate(f'Meilleur:\nS{best_subset+1}\n{all_subsets[best_subset]}',
            xy=(best_subset, align_by_subset[best_subset]),
            xytext=(best_subset + 2, align_by_subset[best_subset] + 0.005),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
            fontsize=8, color='black')
ax.text(3, ax.get_ylim()[0]*1.01, 'Non-overlap', color='#3498db', fontsize=9, ha='center')
ax.text(14, ax.get_ylim()[0]*1.01, 'Random overlapping', color='#e74c3c', fontsize=9, ha='center')

# Figure 2 : heatmap alignment par (subset, famille)
ax = axes[1]
im = ax.imshow(align_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xlabel('Famille de kernel', fontsize=11)
ax.set_ylabel('Sous-ensemble $S_m$', fontsize=11)
ax.set_xticks(range(n_fam))
ax.set_xticklabels(fam_names, fontsize=9)
ax.set_yticks(range(n_sub))
ax.set_yticklabels([f'S{i+1}' for i in range(n_sub)], fontsize=8)
ax.set_title('Alignment par (subset, famille)\nplus chaud = plus aligné', fontweight='bold')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

# Top-10 kernels par alignment
top_idx = np.argsort(q)[::-1][:10]
print("\nTop 10 kernels par alignment avec la cible :")
print(f"{'Rang':>4} {'Kernel':30} {'Features':20} {'Alignment':>10}")
print("-"*70)
for rank, idx in enumerate(top_idx, 1):
    name = kernel_names[idx]
    s_idx, fname, alpha = kernel_meta[name]
    feat_str = str(all_subsets[s_idx])
    print(f"  {rank:2d}. {name:30s} {feat_str:20s} {q[idx]:.5f}")


## Section 5 — Comparaison AUC : baseline vs sous-ensembles différents

On compare 5 configurations en **AUC moyen** (5-fold stratified CV, SVM kernel precomputed, C=1.0) :

| Configuration | Nbre kernels | Description |
|---|---|---|
| **Baseline** | 1 | Features {0,1,2,3} fixes — implémentation actuelle |
| **QMKL Non-overlap** | 28 | 7 subsets × 4 familles — couvre 28 features |
| **QMKL All** | 88 | 22 subsets × 4 familles — couvre tout l'espace |
| **QMKL Top-10** | 10 | Meilleurs par alignment (poids proportionnels à $q_m$) |
| **RBF-SVM** | — | Référence classique ($\gamma$=scale, C=1.0) |

**Attention** : le SVM avec kernel precomputed nécessite de passer la bonne
sous-matrice à chaque fold (train×train pour l'entraînement, test×train pour l'inférence).


In [ ]:
# ── Cross-validation avec kernel precomputed ──────────────────────────────

def cv_auc_precomputed(K, y, n_splits=5, C=1.0, seed=42):
    '''AUC moyen en cross-validation avec kernel precomputed.
    Passe K_train[train,train] pour fit et K_test[test,train] pour predict.'''
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    aucs = []
    for train_idx, test_idx in kf.split(K, y):
        K_train = K[np.ix_(train_idx, train_idx)]
        K_test  = K[np.ix_(test_idx,  train_idx)]
        y_train, y_test = y[train_idx], y[test_idx]
        clf = SVC(kernel='precomputed', C=C, probability=True)
        clf.fit(K_train, y_train)
        proba = clf.predict_proba(K_test)[:, 1]
        aucs.append(roc_auc_score(y_test, proba))
    return np.mean(aucs), np.std(aucs)

def combine_kernels(kernel_list, weights=None):
    '''Combinaison pondérée de kernels (uniforme si weights=None).'''
    if weights is None:
        weights = np.ones(len(kernel_list)) / len(kernel_list)
    w = np.array(weights, dtype=float)
    w = w / w.sum()
    return sum(wi * Ki for wi, Ki in zip(w, kernel_list))

# ── Config 1 : Baseline ────────────────────────────────────────────────────
K_base = kernels['S1_ZZ_a20']    # features {0,1,2,3}, ZZ alpha=2.0
auc_base, std_base = cv_auc_precomputed(K_base, y)
print(f"Baseline (features {{0,1,2,3}}, ZZ α=2.0) : AUC = {auc_base:.4f} ± {std_base:.4f}")

# ── Config 2 : QMKL Non-overlapping (7 subsets × 4 familles = 28 kernels) ─
names_nonop = [f'S{i+1}_{f}' for i in range(7) for f in FAMILIES]
K_nonop = combine_kernels([kernels[n] for n in names_nonop])
auc_nonop, std_nonop = cv_auc_precomputed(K_nonop, y)
print(f"QMKL Non-overlap  (7 subsets, 28K)       : AUC = {auc_nonop:.4f} ± {std_nonop:.4f}")

# ── Config 3 : QMKL All (22 subsets × 4 familles = 88 kernels) ────────────
K_all = combine_kernels(list(kernels.values()))
auc_all, std_all = cv_auc_precomputed(K_all, y)
print(f"QMKL All          (22 subsets, 88K)       : AUC = {auc_all:.4f} ± {std_all:.4f}")

# ── Config 4 : QMKL Top-10 par alignment ──────────────────────────────────
top10_K = [kernels[kernel_names[i]] for i in top_idx]
top10_w = [q[i] for i in top_idx]
K_top10 = combine_kernels(top10_K, weights=top10_w)
auc_top10, std_top10 = cv_auc_precomputed(K_top10, y)
print(f"QMKL Top-10       (alignment-weighted)    : AUC = {auc_top10:.4f} ± {std_top10:.4f}")

# ── Config 5 : RBF-SVM ────────────────────────────────────────────────────
from sklearn.model_selection import cross_val_score
clf_rbf = SVC(kernel='rbf', C=1.0, probability=True)
kf5 = StratifiedKFold(5, shuffle=True, random_state=42)
aucs_rbf = cross_val_score(clf_rbf, X, y, cv=kf5, scoring='roc_auc')
auc_rbf, std_rbf = aucs_rbf.mean(), aucs_rbf.std()
print(f"RBF-SVM           (gamma=scale, C=1.0)    : AUC = {auc_rbf:.4f} ± {std_rbf:.4f}")

# ── Figure comparative ─────────────────────────────────────────────────────
labels  = ['Baseline\n(4 features fixes)', 'QMKL\nNon-overlap\n(28K)',
           'QMKL All\n(88K)', 'QMKL Top-10\n(alignment)', 'RBF-SVM\n(référence)']
aucs_  = [auc_base, auc_nonop, auc_all, auc_top10, auc_rbf]
stds_  = [std_base, std_nonop, std_all, std_top10, std_rbf]
colors = ['#bdc3c7', '#3498db', '#2980b9', '#e74c3c', '#2c3e50']

fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(labels))
bars = ax.bar(x, aucs_, yerr=stds_, capsize=6, color=colors, edgecolor='white', linewidth=0.5, width=0.6)

for bar, auc, std in zip(bars, aucs_, stds_):
    ax.text(bar.get_x() + bar.get_width()/2, auc + std + 0.0008,
            f'{auc:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(auc_rbf, color='#2c3e50', linestyle='--', linewidth=1.5, alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('AUC (5-fold stratified CV)', fontsize=12)
ax.set_title('Breast Cancer — QMKL avec sous-ensembles de features vs baseline\n'
             'SVM kernel precomputed, C=1.0', fontweight='bold')

ymin = min(aucs_) - 3*max(stds_)
ax.set_ylim(max(0.93, ymin), 1.005)
plt.tight_layout()
plt.show()

# ── Résumé chiffré ─────────────────────────────────────────────────────────
print("\n=== Résumé des gains ===")
print(f"Non-overlap vs Baseline : {(auc_nonop-auc_base)*100:+.2f} pts AUC")
print(f"All subsets vs Baseline : {(auc_all-auc_base)*100:+.2f} pts AUC")
print(f"Top-10 vs Baseline      : {(auc_top10-auc_base)*100:+.2f} pts AUC")
print(f"Top-10 vs RBF-SVM       : {(auc_top10-auc_rbf)*100:+.2f} pts AUC")
print(f"\n→ Couvrir plus de features aide (ou non) selon la structure du dataset.")
print(f"→ Le Top-10 par alignment est souvent meilleur que la moyenne uniforme.")


## Section 6 — Importance implicite des features

### Principe

Chaque kernel $K_m$ utilise les 4 features de $S_m$.
L'importance d'une feature $k$ est la **somme des poids** de tous les kernels qui l'incluent :

$$\text{importance}(k) = \sum_{m : k \in S_m} q_m$$

où $q_m = \hat{A}(K_m, \mathbf{y}\mathbf{y}^\top)$ est l'alignment du kernel $m$ avec la cible.

### Interprétation

Cette importance **n'est pas supervisée au sens classique** : on n'a pas régressé
la cible $y$ sur les features directement. QMKL a exploré la géométrie quantique
de l'espace et remonté les features qui permettent de discriminer.

On compare avec la corrélation de Pearson $|\rho(x_k, y)|$,
qui est le test de feature importance le plus simple possible.


In [ ]:
# ── Importance implicite des features via les poids d'alignment ───────────

# Poids normalisés (proportionnels à l'alignment)
w_norm = q / q.sum()

# importance(k) = somme des poids des kernels qui contiennent la feature k
feature_importance = np.zeros(d)
for name, w in zip(kernel_names, w_norm):
    s_idx, fname, alpha = kernel_meta[name]
    for k in all_subsets[s_idx]:
        feature_importance[k] += w

# Normaliser pour la visualisation
feat_imp_norm = feature_importance / feature_importance.max()

# ── Références classiques ──────────────────────────────────────────────────
# Corrélation absolue de Pearson avec la cible
correlations = np.array([np.abs(np.corrcoef(X_raw[:, k], y_pm)[0, 1]) for k in range(d)])

# Variance des features (normalisée)
feat_std = X_raw.std(axis=0)
feat_std_norm = feat_std / feat_std.max()

# ── Figure : 3 métriques en parallèle ─────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
x = np.arange(d)

short_names = []
for n in feature_names:
    n = n.replace(' (SE)', ' SE').replace('mean ', 'mean\n').replace('worst ', 'worst\n')
    short_names.append(n[:16])

axes[0].bar(x, feat_imp_norm, color='#3498db', alpha=0.85, edgecolor='white')
axes[0].set_ylabel('Importance QMKL\n(normalisée)', fontsize=10)
axes[0].set_title('Importance implicite via Centered Alignment QMKL', fontweight='bold')

axes[1].bar(x, correlations, color='#e74c3c', alpha=0.85, edgecolor='white')
axes[1].set_ylabel('|Corrélation|\navec target', fontsize=10)
axes[1].set_title('Corrélation de Pearson avec la cible (référence classique)', fontweight='bold')

axes[2].bar(x, feat_std_norm, color='#2ecc71', alpha=0.85, edgecolor='white')
axes[2].set_ylabel('Écart-type\n(normalisé)', fontsize=10)
axes[2].set_title('Variance des features (brutes, non-normalisées)', fontweight='bold')

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()

# ── Corrélation de Spearman entre importance QMKL et corrélation classique ─
rho, pval = spearmanr(feat_imp_norm, correlations)
print(f"Corrélation de Spearman (QMKL importance vs |corr target|) : rho={rho:.3f}, p={pval:.4f}")

# ── Scatter : importance QMKL vs corrélation classique ────────────────────
fig, ax = plt.subplots(figsize=(9, 5.5))
sc = ax.scatter(correlations, feat_imp_norm,
                c=feat_std_norm, cmap='viridis', s=70, alpha=0.85,
                edgecolors='white', linewidth=0.5)
plt.colorbar(sc, ax=ax, label='Variance normalisée')
ax.set_xlabel('|Corrélation de Pearson| avec la cible', fontsize=11)
ax.set_ylabel('Importance QMKL (Centered Alignment)', fontsize=11)
ax.set_title(f'QMKL retrouve-t-il les features importantes ?\n'
             f'Corrélation de Spearman : rho={rho:.2f}', fontweight='bold')

# Annoter les 5 meilleures par QMKL
top5_qmkl = np.argsort(feat_imp_norm)[::-1][:5]
for k in top5_qmkl:
    ax.annotate(feature_names[k][:20],
                (correlations[k], feat_imp_norm[k]),
                fontsize=7, textcoords='offset points', xytext=(6, 3),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.0))

# Ligne de tendance
z = np.polyfit(correlations, feat_imp_norm, 1)
xfit = np.linspace(correlations.min(), correlations.max(), 100)
ax.plot(xfit, np.poly1d(z)(xfit), 'r--', alpha=0.5, label=f'Tendance linéaire (rho={rho:.2f})')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── Tableau de comparaison Top-5 ───────────────────────────────────────────
top5_corr = np.argsort(correlations)[::-1][:5]
top5_qmkl = np.argsort(feat_imp_norm)[::-1][:5]
overlap = set(top5_qmkl) & set(top5_corr)

print("\nTop 5 features par importance QMKL :")
for rank, k in enumerate(top5_qmkl, 1):
    print(f"  {rank}. {feature_names[k]:35s} importance={feat_imp_norm[k]:.3f}, |corr|={correlations[k]:.3f}")

print("\nTop 5 features par |corrélation| classique :")
for rank, k in enumerate(top5_corr, 1):
    print(f"  {rank}. {feature_names[k]:35s} |corr|={correlations[k]:.3f}, importance={feat_imp_norm[k]:.3f}")

print(f"\nFeatures communes top-5 : {len(overlap)}/5")
if overlap:
    for k in overlap:
        print(f"  → {feature_names[k]}")

print("\nNote : les features absentes des subsets non-overlapping (features 28-29)")
print(f"  ont importance=0 si elles n'apparaissent que dans les random subsets peu alignés.")


## Conclusion — Ce qu'apporte l'approche par sous-ensembles

### Ce qu'on a montré

| Résultat | Valeur |
|---|---|
| Espace implicite avec M=22 subsets, Q=4 | 352 dimensions quantiques |
| Gain AUC (non-overlap vs 4 fixes) | voir Section 5 |
| Cohérence importance QMKL / corrélation | voir $\rho$ de Spearman |

### Les 3 apports clés

**1. Scalabilité : $d \gg Q$ n'est plus un problème**

Au lieu de PCA-4 (arbitraire), on couvre l'espace systématiquement.
La limite n'est plus $d = Q$ mais $d = M \times Q$ avec M choisi librement.

**2. Sélection de features implicite**

Les poids $w_m$ révèlent quels groupes de features sont discriminants,
**sans nécessiter de feature selection supervisée explicite**.
C'est une byproduct gratuite du Centered Alignment.

**3. Diversité structurelle**

Des kernels sur des subsets différents sont par construction
moins redondants que des kernels sur les mêmes features.
L'alignment mutuel $S_{mn}$ est naturellement plus faible.

### Limites honnêtes

- **Coût** : $M \times N^2$ opérations pour calculer tous les kernels
- **Choix des subsets** : aléatoire ici — des méthodes informées (corrélation, PCA locale) peuvent faire mieux
- **Bruit hardware** : en vrai quantum, chaque circuit supplémentaire ajoute du bruit — avec $M=88$ circuits, le bruit se propage
- **Breast Cancer** est un dataset où RBF-SVM est quasi-optimal : les gains QMKL ici montrent la mécanique, pas une supériorité quantique

### Quand utiliser cette approche

| Condition | Action |
|---|---|
| $d > Q$ (toujours en pratique) | Subsets non-overlapping en premier |
| Features corrélées par groupes | Subsets définis par corrélation (groupes de features liées) |
| Interprétabilité souhaitée | Poids d'alignment = importance des features |
| Hardware quantique disponible | M petit, subsets de haute qualité par alignment |
| $N$ grand ($N > 1000$) | Calcul de $M \times N^2$ kernels coûteux — sélectionner M < 20 |
